In [59]:
import pandas as pd
import duckdb
import seaborn as sns

In [60]:
sns.set_theme()
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)

In [61]:
con = duckdb.connect("../data/raw_layer.db")

In [62]:
con.execute("SHOW ALL TABLES").fetchdf()

,database,schema,name,column_names,column_types,temporary
0,raw_layer,main,raw_events,"[id, name, deadline_time, release_time, averag...","[BIGINT, VARCHAR, VARCHAR, INTEGER, BIGINT, BO...",False
1,raw_layer,main,raw_fixtures,"[code, event, finished, finished_provisional, ...","[BIGINT, BIGINT, BOOLEAN, BOOLEAN, BIGINT, VAR...",False
2,raw_layer,main,raw_gw_data,"[id, explain, modified, stats.minutes, stats.g...","[DOUBLE, STRUCT(fixture INTEGER, stats STRUCT(...",False
3,raw_layer,main,raw_log,"[id, payload, response_code, table_name, times...","[INTEGER, JSON, VARCHAR, VARCHAR, TIMESTAMP, V...",False
4,raw_layer,main,raw_players,"[can_transact, can_select, chance_of_playing_n...","[BOOLEAN, BOOLEAN, DOUBLE, DOUBLE, BIGINT, BIG...",False
5,raw_layer,main,raw_teams,"[code, draw, form, id, loss, name, played, poi...","[BIGINT, BIGINT, INTEGER, BIGINT, BIGINT, VARC...",False


## Raw Events

In [63]:
raw_events = con.execute("SELECT * FROM raw_events").fetchdf()
print(raw_events.columns)

Index(['id', 'name', 'deadline_time', 'release_time', 'average_entry_score',
       'finished', 'data_checked', 'highest_scoring_entry',
       'deadline_time_epoch', 'deadline_time_game_offset', 'highest_score',
       'is_previous', 'is_current', 'is_next', 'cup_leagues_created',
       'h2h_ko_matches_created', 'can_enter', 'can_manage', 'released',
       'ranked_count', 'overrides', 'chip_plays', 'most_selected',
       'most_transferred_in', 'top_element', 'top_element_info',
       'transfers_made', 'most_captained', 'most_vice_captained',
       'ingested_at'],
      dtype='str')


In [64]:
raw_events.head()

,id,name,deadline_time,release_time,average_entry_score,finished,data_checked,highest_scoring_entry,deadline_time_epoch,deadline_time_game_offset,highest_score,is_previous,is_current,is_next,cup_leagues_created,h2h_ko_matches_created,can_enter,can_manage,released,ranked_count,overrides,chip_plays,most_selected,most_transferred_in,top_element,top_element_info,transfers_made,most_captained,most_vice_captained,ingested_at
0,1,Gameweek 1,2025-08-15T17:30:00Z,<NA>,54,True,True,3772644.0,1755279000,0,127.0,False,False,False,False,False,False,False,True,9469118,"{'rules': {}, 'scoring': {}, 'element_types': ...","[{'chip_name': 'bboost', 'num_played': 342779}...",235.0,1.0,531.0,"{'id': 531, 'points': 17}",0,381.0,235.0,2026-04-08 22:28:26.848273
1,2,Gameweek 2,2025-08-22T17:30:00Z,<NA>,51,True,True,2963810.0,1755883800,0,140.0,False,False,False,True,True,False,False,True,10752422,"{'rules': {}, 'scoring': {}, 'element_types': ...","[{'chip_name': 'bboost', 'num_played': 206501}...",235.0,427.0,8.0,"{'id': 8, 'points': 24}",18178809,381.0,235.0,2026-04-08 22:28:26.848273
2,3,Gameweek 3,2025-08-30T10:00:00Z,<NA>,48,True,True,4374360.0,1756548000,0,118.0,False,False,False,True,True,False,False,True,11304791,"{'rules': {}, 'scoring': {}, 'element_types': ...","[{'chip_name': 'bboost', 'num_played': 181694}...",249.0,82.0,260.0,"{'id': 260, 'points': 15}",27802596,430.0,381.0,2026-04-08 22:28:26.848273
3,4,Gameweek 4,2025-09-13T10:00:00Z,<NA>,63,True,True,7997042.0,1757757600,0,139.0,False,False,False,True,True,False,False,True,11641518,"{'rules': {}, 'scoring': {}, 'element_types': ...","[{'chip_name': 'bboost', 'num_played': 151177}...",249.0,419.0,26.0,"{'id': 26, 'points': 16}",27408255,381.0,249.0,2026-04-08 22:28:26.848273
4,5,Gameweek 5,2025-09-20T10:00:00Z,<NA>,42,True,True,9985225.0,1758362400,0,112.0,False,False,False,True,True,False,False,True,11803614,"{'rules': {}, 'scoring': {}, 'element_types': ...","[{'chip_name': 'bboost', 'num_played': 231274}...",249.0,82.0,660.0,"{'id': 660, 'points': 15}",16533109,381.0,249.0,2026-04-08 22:28:26.848273


In [65]:
print(raw_events.shape)

(38, 30)


In [66]:
print(raw_events.info())

<class 'pandas.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 30 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   id                         38 non-null     int64         
 1   name                       38 non-null     str           
 2   deadline_time              38 non-null     str           
 3   release_time               0 non-null      Int32         
 4   average_entry_score        38 non-null     int64         
 5   finished                   38 non-null     bool          
 6   data_checked               38 non-null     bool          
 7   highest_scoring_entry      31 non-null     float64       
 8   deadline_time_epoch        38 non-null     int64         
 9   deadline_time_game_offset  38 non-null     int64         
 10  highest_score              31 non-null     float64       
 11  is_previous                38 non-null     bool          
 12  is_current           

### Findings:
- Each row is a gameweek and the fields give us a high level overview of managerial decisions
- highest_score, most_selected, most_transferred_in, top_element, top_element_info, most_captained, and most_vice_captained only have values for games that have been played already
- **We can use deadline_time to schedule Airflow runs**
- Nothing else seems super relevant for predicting the order to play players in
- *Maybe records of Wins/Losses for each team?*
- This table seems mostly useful just to tell the record of teams against each other and to keep track of gameweeks, could potentially use to calculate differential moves

## Raw Fixtures Data

In [67]:
raw_fixtures = con.execute("SELECT * FROM raw_fixtures").fetchdf()
print(raw_fixtures.columns)

Index(['code', 'event', 'finished', 'finished_provisional', 'id',
       'kickoff_time', 'minutes', 'provisional_start_time', 'started',
       'team_a', 'team_a_score', 'team_h', 'team_h_score', 'stats',
       'team_h_difficulty', 'team_a_difficulty', 'pulse_id', 'gameweek'],
      dtype='str')


In [85]:
raw_fixtures.tail()

,code,event,finished,finished_provisional,id,kickoff_time,minutes,provisional_start_time,started,team_a,team_a_score,team_h,team_h_score,stats,team_h_difficulty,team_a_difficulty,pulse_id,gameweek
374,2562270,38,False,False,376,2026-05-24T15:00:00Z,0,False,False,2,<NA>,13,<NA>,[],3,5,125166,38
375,2562271,38,False,False,377,2026-05-24T15:00:00Z,0,False,False,4,<NA>,16,<NA>,[],3,3,125167,38
376,2562273,38,False,False,379,2026-05-24T15:00:00Z,0,False,False,9,<NA>,18,<NA>,[],3,2,125169,38
377,2562272,38,False,False,378,2026-05-24T15:00:00Z,0,False,False,7,<NA>,17,<NA>,[],3,3,125168,38
378,2562274,38,False,False,380,2026-05-24T15:00:00Z,0,False,False,11,<NA>,19,<NA>,[],2,2,125170,38


In [69]:
print(raw_fixtures.shape)

(379, 18)


In [70]:
print(raw_fixtures.info())

<class 'pandas.DataFrame'>
RangeIndex: 379 entries, 0 to 378
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   code                    379 non-null    int64 
 1   event                   379 non-null    int64 
 2   finished                379 non-null    bool  
 3   finished_provisional    379 non-null    bool  
 4   id                      379 non-null    int64 
 5   kickoff_time            379 non-null    str   
 6   minutes                 379 non-null    int64 
 7   provisional_start_time  379 non-null    bool  
 8   started                 379 non-null    bool  
 9   team_a                  379 non-null    int64 
 10  team_a_score            309 non-null    Int32 
 11  team_h                  379 non-null    int64 
 12  team_h_score            309 non-null    Int32 
 13  stats                   379 non-null    object
 14  team_h_difficulty       379 non-null    int64 
 15  team_a_difficulty

In [71]:
stats_parsed = raw_fixtures['stats'].to_dict()
print(len(stats_parsed))

379


### Relevant Fields
- **event** = Gameweek
- **finished** = True if match is over and Bonus points have been attributed
- **finished_provisional** = True if game has been played, False if not
- **kickoff_time** = Timestamp of the kickoff
- **team_a** = Away team ID, maps to a name
- **team_a**_score = Goals scored by Away team
- **team_h** = Home team ID, maps to a name
- **team_h_score** = Goals scored by Home team
- **stats**: nested field **can be parsed further to get goals_scored, assists, saves, defcons and bps** -- Edit: This is all parsed in the Raw Gameweek Data below alread
- **team_h_difficulty** = Difficulty for the Home team
- **team_a_difficulty** = Difficulty for the Away team

### Findings:
- This is every game that is scheduled in the Prem
- Contains all stats within the game, as well as what team they are playing next
- Can use the existing FDR for calculating difficulty of a fixture, *or can calculate our own*
- Home and Away scores as well as stats per game are calculated as the gameweeks progress

## Raw Gameweek Data

In [72]:
raw_gw_data = con.execute("SELECT * FROM raw_gw_data").fetchdf()
print(raw_gw_data.columns)

Index(['id', 'explain', 'modified', 'stats.minutes', 'stats.goals_scored',
       'stats.assists', 'stats.clean_sheets', 'stats.goals_conceded',
       'stats.own_goals', 'stats.penalties_saved', 'stats.penalties_missed',
       'stats.yellow_cards', 'stats.red_cards', 'stats.saves', 'stats.bonus',
       'stats.bps', 'stats.influence', 'stats.creativity', 'stats.threat',
       'stats.ict_index', 'stats.clearances_blocks_interceptions',
       'stats.recoveries', 'stats.tackles', 'stats.defensive_contribution',
       'stats.starts', 'stats.expected_goals', 'stats.expected_assists',
       'stats.expected_goal_involvements', 'stats.expected_goals_conceded',
       'stats.total_points', 'stats.in_dreamteam', 'gameweek'],
      dtype='str')


In [73]:
raw_gw_data.head()

,id,explain,modified,stats.minutes,stats.goals_scored,stats.assists,stats.clean_sheets,stats.goals_conceded,stats.own_goals,stats.penalties_saved,stats.penalties_missed,stats.yellow_cards,stats.red_cards,stats.saves,stats.bonus,stats.bps,stats.influence,stats.creativity,stats.threat,stats.ict_index,stats.clearances_blocks_interceptions,stats.recoveries,stats.tackles,stats.defensive_contribution,stats.starts,stats.expected_goals,stats.expected_assists,stats.expected_goal_involvements,stats.expected_goals_conceded,stats.total_points,stats.in_dreamteam,gameweek
0,1.0,"[{'fixture': 9, 'stats': [{'identifier': 'minu...",False,90.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,7.0,3.0,38.0,49.2,0.0,0.0,4.9,1.0,13.0,0.0,0.0,1.0,0.00,0.00,0.00,1.52,10.0,True,1
1,2.0,"[{'fixture': 9, 'stats': [{'identifier': 'minu...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0,False,1
2,3.0,"[{'fixture': 9, 'stats': [{'identifier': 'minu...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0,False,1
3,4.0,"[{'fixture': 9, 'stats': [{'identifier': 'minu...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0,False,1
4,5.0,"[{'fixture': 9, 'stats': [{'identifier': 'minu...",False,90.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,23.0,22.8,0.3,4.0,2.7,6.0,1.0,1.0,7.0,1.0,0.00,0.00,0.00,1.52,6.0,False,1


In [87]:
raw_gw_data['explain'][0]

array([{'fixture': 9, 'stats': [{'identifier': 'minutes', 'points': 2, 'value': 90, 'points_modification': 0}, {'identifier': 'clean_sheets', 'points': 4, 'value': 1, 'points_modification': 0}, {'identifier': 'yellow_cards', 'points': -1, 'value': 1, 'points_modification': 0}, {'identifier': 'saves', 'points': 2, 'value': 7, 'points_modification': 0}, {'identifier': 'bonus', 'points': 3, 'value': 3, 'points_modification': 0}]}],
      dtype=object)

In [74]:
print(raw_gw_data.shape)

(23911, 32)


In [75]:
print(raw_gw_data.info())

<class 'pandas.DataFrame'>
RangeIndex: 23911 entries, 0 to 23910
Data columns (total 32 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id                                     23911 non-null  float64
 1   explain                                23911 non-null  object 
 2   modified                               23911 non-null  bool   
 3   stats.minutes                          23911 non-null  float64
 4   stats.goals_scored                     23911 non-null  float64
 5   stats.assists                          23911 non-null  float64
 6   stats.clean_sheets                     23911 non-null  float64
 7   stats.goals_conceded                   23911 non-null  float64
 8   stats.own_goals                        23911 non-null  float64
 9   stats.penalties_saved                  23911 non-null  float64
 10  stats.penalties_missed                 23911 non-null  float64
 11  stats.yellow_

### Findings:
- Looks like this table contains all the stats data parsed from the Fixtures data
- Each row is a player(id)'s stats in a given game
- 21 players per team * 38 gameweeks * 10 matches per week * 2 teams per match = ~16k
- The diff in row count ~24k - ~16k is probably accounted for by transferred/reserve players who are added to the roster
- **This is the core table we will be using in our analysis**

## Raw Players

In [76]:
raw_players = con.execute("SELECT * FROM raw_players").fetchdf()
print(raw_players.columns.tolist())

['can_transact', 'can_select', 'chance_of_playing_next_round', 'chance_of_playing_this_round', 'code', 'cost_change_event', 'cost_change_event_fall', 'cost_change_start', 'cost_change_start_fall', 'price_change_percent', 'dreamteam_count', 'element_type', 'ep_next', 'ep_this', 'event_points', 'first_name', 'form', 'id', 'in_dreamteam', 'news', 'news_added', 'now_cost', 'photo', 'points_per_game', 'removed', 'second_name', 'selected_by_percent', 'special', 'squad_number', 'status', 'team', 'team_code', 'total_points', 'transfers_in', 'transfers_in_event', 'transfers_out', 'transfers_out_event', 'value_form', 'value_season', 'web_name', 'known_name', 'region', 'team_join_date', 'birth_date', 'has_temporary_code', 'opta_code', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity', 'threat', 'ict_index', 'clearances_blocks_interceptions', 're

In [77]:
raw_players.head()

,can_transact,can_select,chance_of_playing_next_round,chance_of_playing_this_round,code,cost_change_event,cost_change_event_fall,cost_change_start,cost_change_start_fall,price_change_percent,dreamteam_count,element_type,ep_next,ep_this,event_points,first_name,form,id,in_dreamteam,news,news_added,now_cost,photo,points_per_game,removed,second_name,selected_by_percent,special,squad_number,status,team,team_code,total_points,transfers_in,transfers_in_event,transfers_out,transfers_out_event,value_form,value_season,web_name,known_name,region,team_join_date,birth_date,has_temporary_code,opta_code,minutes,goals_scored,assists,clean_sheets,goals_conceded,own_goals,penalties_saved,penalties_missed,yellow_cards,red_cards,saves,bonus,bps,influence,creativity,threat,ict_index,clearances_blocks_interceptions,recoveries,tackles,defensive_contribution,starts,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,corners_and_indirect_freekicks_order,corners_and_indirect_freekicks_text,direct_freekicks_order,direct_freekicks_text,penalties_order,penalties_text,scout_risks,scout_news_link,influence_rank,influence_rank_type,creativity_rank,creativity_rank_type,threat_rank,threat_rank_type,ict_index_rank,ict_index_rank_type,expected_goals_per_90,saves_per_90,expected_assists_per_90,expected_goal_involvements_per_90,expected_goals_conceded_per_90,goals_conceded_per_90,now_cost_rank,now_cost_rank_type,form_rank,form_rank_type,points_per_game_rank,points_per_game_rank_type,selected_rank,selected_rank_type,starts_per_90,clean_sheets_per_90,defensive_contribution_per_90,ingested_at
0,True,True,NaN,NaN,154561,0,0,5,-5,0,1,1,8.0,0.0,0,David,7.0,1,True,,NaN,60,154561.jpg,4.2,False,Raya Martín,34.6,False,<NA>,a,1,3,129,4193496,72819,2511130,43836,1.2,21.5,Raya,,200.0,2024-07-04,1995-09-15,False,p154561,2790,0,0,15,22,0,0,0,1,0,50,6,515,445.8,33.4,0.0,48.0,30,257,1,0,31,0.00,0.06,0.06,22.30,NaN,,NaN,,NaN,,[],,108,18,376,3,816,95,270,16,0.00,1.61,0.00,0.00,0.72,0.71,85,1,24,2,51,3,9,1,1.0,0.48,0.00,2026-04-08 22:28:26.826789
1,True,True,NaN,NaN,109745,0,0,-5,5,0,0,1,1.0,0.0,0,Kepa,0.0,2,False,,NaN,40,109745.jpg,0.0,False,Arrizabalaga Revuelta,0.4,False,<NA>,a,1,3,0,14982,230,77658,249,0.0,0.0,Arrizabalaga,,200.0,2025-07-01,1994-10-03,False,p109745,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,0,0.00,0.00,0.00,0.00,NaN,,NaN,,NaN,,[],,570,66,560,52,530,39,572,66,0.00,0.00,0.00,0.00,0.00,0.00,667,43,406,53,581,66,288,37,0.0,0.00,0.00,2026-04-08 22:28:26.826789
2,True,False,0.0,0.0,463748,0,0,0,0,0,0,1,0.0,0.0,0,Karl,0.0,3,False,Has joined Werder Bremen on loan for the rest ...,2025-08-26T13:44:02.357864Z,40,463748.jpg,0.0,False,Hein,0.2,False,<NA>,u,1,3,0,5545,0,48359,61,0.0,0.0,Hein,,67.0,2020-10-28,2002-04-13,False,p463748,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,0,0.00,0.00,0.00,0.00,NaN,,NaN,,NaN,,[],,587,75,578,62,548,49,589,75,0.00,0.00,0.00,0.00,0.00,0.00,687,53,424,63,598,75,375,54,0.0,0.00,0.00,2026-04-08 22:28:26.826789
3,True,True,NaN,NaN,551221,0,0,-1,1,0,0,1,1.0,0.0,0,Tommy,0.0,4,False,,NaN,39,551221.jpg,0.0,False,Setford,0.2,False,<NA>,a,1,3,0,33979,65,34321,219,0.0,0.0,Setford,,241.0,2024-07-24,2006-03-13,False,p551221,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,0,0.00,0.00,0.00,0.00,NaN,,NaN,,NaN,,[],,559,60,547,44,516,31,561,60,0.00,0.00,0.00,0.00,0.00,0.00,778,89,393,46,570,60,364,52,0.0,0.00,0.00,2026-04-08 22:28:26.826789
4,True,True,100.0,100.0,226597,0,0,12,-12,0,5,2,10.0,0.0,0,Gabriel,9.0,5,True,,2026-03-24T00:00:10.555559Z,72,226597.jpg,6.9,False,dos Santos Magalhães,42.9,False,<NA>,a,1,3,173,8355443,92103,5464894,78370,1.2,24.0,Gabriel,Gabriel Magalhães,30.0,2020-09-01,1997-12-19,False,p226597,2165,3,4,14,15,0,0,0,3,0,0,25,593,686.8,108.4,199.0,99.2,192,46,32,224,24,1.90,1.65,3.55,15.82,NaN,,NaN,,NaN,,[],,26,9,262,74,142,27,94,25,0.08,0.00,0.07,0.15,0.66,0.62,32,1,11,7,2,1,5,1,1.0,0.58,9.31,2026-04-08 22:28:26.826789


In [78]:
print(raw_players.shape)

(825, 106)


In [79]:
print(raw_players.info(verbose=True, show_counts=True))

<class 'pandas.DataFrame'>
RangeIndex: 825 entries, 0 to 824
Data columns (total 106 columns):
 #    Column                                Non-Null Count  Dtype         
---   ------                                --------------  -----         
 0    can_transact                          825 non-null    bool          
 1    can_select                            825 non-null    bool          
 2    chance_of_playing_next_round          612 non-null    float64       
 3    chance_of_playing_this_round          605 non-null    float64       
 4    code                                  825 non-null    int64         
 5    cost_change_event                     825 non-null    int64         
 6    cost_change_event_fall                825 non-null    int64         
 7    cost_change_start                     825 non-null    int64         
 8    cost_change_start_fall                825 non-null    int64         
 9    price_change_percent                  825 non-null    str           
 10  

## Relevant Fields
- id
- first_name
- second_name
- news
- news_added
- now_cost
- form
- points_per_game
- team
- team_code?
- total_points
- transfers_in
- transfers_out
- minutes?
- goals_scored        
- assists       
- clean_sheets        
- goals_conceded      
- own_goals       
- penalties_saved       
- penalties_missed       
- yellow_cards     
- red_cards        
- saves      
- bonus
- bps
- influence     
- creativity
- threat



**This might not be a good table to train ML on due to time series leakage. It might be easier to use create the training data for these fields next season and train a new model**

## Findings
- This is the overall stats of every player in the Premier League
- Chance of playing game should be custom calculated as well(Use LLM to parse news field?)
- Set piece fields might not be the most accurate, might be worth it to pull from https://www.fantasyfootballscout.co.uk/fantasy-premier-league-set-piece-takers

## Raw Teams

In [80]:
raw_teams = con.execute("SELECT * FROM raw_teams").fetchdf()
print(raw_teams.columns)

Index(['code', 'draw', 'form', 'id', 'loss', 'name', 'played', 'points',
       'position', 'short_name', 'strength', 'team_division', 'unavailable',
       'win', 'strength_overall_home', 'strength_overall_away',
       'strength_attack_home', 'strength_attack_away', 'strength_defence_home',
       'strength_defence_away', 'pulse_id', 'ingested_at'],
      dtype='str')


In [81]:
raw_teams.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   code                   20 non-null     int64         
 1   draw                   20 non-null     int64         
 2   form                   0 non-null      Int32         
 3   id                     20 non-null     int64         
 4   loss                   20 non-null     int64         
 5   name                   20 non-null     str           
 6   played                 20 non-null     int64         
 7   points                 20 non-null     int64         
 8   position               20 non-null     int64         
 9   short_name             20 non-null     str           
 10  strength               20 non-null     int64         
 11  team_division          0 non-null      Int32         
 12  unavailable            20 non-null     bool          
 13  win               

In [82]:
print(raw_teams.shape)

(20, 22)


In [88]:
raw_teams.head()

,code,draw,form,id,loss,name,played,points,position,short_name,strength,team_division,unavailable,win,strength_overall_home,strength_overall_away,strength_attack_home,strength_attack_away,strength_defence_home,strength_defence_away,pulse_id,ingested_at
0,3,0,<NA>,1,0,Arsenal,0,0,1,ARS,5,<NA>,False,0,1305,1365,1340,1390,1270,1340,1,2026-04-08 22:28:26.814121
1,7,0,<NA>,2,0,Aston Villa,0,0,4,AVL,3,<NA>,False,0,1140,1220,1100,1230,1180,1210,2,2026-04-08 22:28:26.814121
2,90,0,<NA>,3,0,Burnley,0,0,19,BUR,2,<NA>,False,0,975,1080,930,1070,1040,1090,43,2026-04-08 22:28:26.814121
3,91,0,<NA>,4,0,Bournemouth,0,0,13,BOU,3,<NA>,False,0,1140,1215,1060,1210,1220,1220,127,2026-04-08 22:28:26.814121
4,94,0,<NA>,5,0,Brentford,0,0,7,BRE,3,<NA>,False,0,1150,1215,1130,1150,1170,1280,130,2026-04-08 22:28:26.814121


## Findings
- This table can be used to map team names to IDs
- Maybe use strength? However it might just be best to use a custom calculated rolling form